[\[GitHub\] Jupyter Notebook](https://github.com/sslastochkin/mcab/blob/main/docs/guide/05_effect_injection.ipynb)
  
# Injecting the Effect  
  
AA simulations (section 3) verify that your test *does not fire* when there is
no real difference between groups.   
**AB simulations** go one step further: they
inject a synthetic effect into the test group and measure how often the test
*correctly fires*. This is the foundation of power estimation and MDE search
(covered in section 6).

`mcab` handles effect injection through **EffectSizer** objects. Every `Designer*`
class has an `effect_sizer` argument that picks which strategy is used:

| `effect_sizer=` | Class | Effect type |
|---|---|---|
| `'percent'` *(default)* | `PercentEffectSizer` | Relative: `+X %` of current value |
| `'const'` | `ConstantEffectSizer` | Absolute: fixed constant added |
| `'function'` | `FunctionEffectSizer` | Arbitrary user-supplied callable |
| `'custom'` | `CustomEffectSizer` | Pre-computed data with effect already applied |

```python
# Effect sizer is chosen at Designer creation time
designer = DesignerIid(target, effect_sizer='percent', seed=42)  # default
designer = DesignerIid(target, effect_sizer='const',   seed=42)

# The numeric effect magnitude is supplied later at simulation time (section 6):
# pvals = designer.calculate_some_power(effect=0.05, ...)
```
The four sizers behave differently depending on the metric type.  
Understanding this distinction is the main goal of this section.

## The four metric types in `mcab`

`mcab` supports four kinds of metrics, and the **same EffectSizer produces
different results for each type**:

| Type | Data shape | Example metric | `AaData` class |
|---|---|---|---|
| **Continuous iid** | 1-D float array | Revenue, session duration | `AaDataIid` |
| **Proportion iid** | 1-D binary (0 / 1) | Conversion flag | `AaDataIid(proportion=True)` |
| **Ratio Continuous** | `(num, denom)` floats/integers | Average check = revenue / orders | `AaDataRatio(proportion=False)` |
| **Ratio Proportion** | `(num, denom)` floats/integers, `num ≤ denom` | CTR = clicks / impressions | `AaDataRatio(proportion=True)` |

Two important internal details affect every sizer applied to ratio data:

> **Ratio-proportion clip.** When `proportion=True`, the effect is applied to
> the numerator, and the result is automatically clipped so that `num ≤ denom`.
> CTR can never exceed 1 even if the injected effect would push it above.
>
> **Stochastic rounding for `ratio='int'`.** Count-based numerators are integers.
> Applying a small multiplicative effect (e.g. `1 × 1.05 = 1.05`) and then
> truncating gives `1` — the effect disappears. `mcab` uses *stochastic
> rounding*: `1.05` becomes `2` with probability `0.05` and `1` with
> probability `0.95`, so the mean numerator shifts correctly on average.
  
Let's generate data we will work with.  

In [1]:
import numpy as np
from mcab import RandomData
from mcab.effects import (
    ConstantEffectSizer,
    PercentEffectSizer,
    FunctionEffectSizer,
    CustomEffectSizer,
)

rd = RandomData(seed=42)

# ── Continuous iid ────────────────────────────────────────────────────────────
data_cont = rd.normal_data(size=10_000, mean=100, std=30)

# ── Proportion iid (binary 0 / 1) ─────────────────────────────────────────────
data_prop = rd.proportion_data(size=10_000, p=0.10)      # 10 % base conversion

# ── Ratio float (average check: revenue / number of orders) ──────────────────
data_avg_num, data_avg_denom = rd.ratio_data_avg_bill(size=10_000)

# ── Ratio int / CTR (clicks / impressions, num ≤ denom) ──────────────────────
data_ctr_num, data_ctr_denom = rd.ratio_data_ctr(size=10_000, ctr=0.05)

print("Baseline metrics")
print(f"  continuous mean :        {data_cont.mean():.2f}")
print(f"  proportion rate :        {data_prop.mean():.4f}")
print(f"  avg check (num/denom):   {data_avg_num.sum() / data_avg_denom.sum():.2f}")
print(f"  CTR   (num/denom):       {data_ctr_num.sum() / data_ctr_denom.sum():.5f}")


Baseline metrics
  continuous mean :        99.69
  proportion rate :        0.1006
  avg check (num/denom):   504.32
  CTR   (num/denom):       0.05065


## `ConstantEffectSizer` — absolute (additive) effect

`ConstantEffectSizer` adds a **fixed constant `E`** to the metric, regardless
of the original observation value.

| Metric type | What happens | Typical framing |
|---|---|---|
| Continuous iid | Every value `+ E` | "Revenue grows by **$5 per user**" |
| Proportion iid | Conversion **rate** shifts by `+E` (individual 0/1 values flipped to match) | "Conversion grows by **+2 percentage points**" |
| Ratio float | **Numerator** `+= E`; denominator unchanged | "Revenue per user grows by **$50**" |
| Ratio int / CTR | Numerator `+= E`, stochastically rounded; clipped at `denom` | "Clicks per user grow by **+0.5**" |

Use `ConstantEffectSizer` when you know the **absolute lift** in the original
metric units.  
  
!!! tip "Usage"
    To use the effect sizer with given effect you should just pass `effect_sizer='const'` and `effect=your_effect` to designer. Below we will just look at examples, reproducing of this code by yourself not needed.

In [2]:
# Instantiate one sizer per metric type
s_cont = ConstantEffectSizer(proportion=False, ratio=False,   seed=42)
s_prop = ConstantEffectSizer(proportion=True,  ratio=False,   seed=42)
s_rfl  = ConstantEffectSizer(proportion=False, ratio='float', seed=42)
s_rint = ConstantEffectSizer(proportion=True,  ratio='int',   seed=42)

# Apply (seed= is forwarded for reproducibility of stochastic steps)
cont_eff = s_cont(data_cont,                       effect=5.0,  seed=42)
prop_eff = s_prop(data_prop,                       effect=0.02, seed=42)   # +2 pp
rfl_eff  = s_rfl( (data_avg_num, data_avg_denom),  effect=50.0, seed=42)   # +50 to numerator
rint_eff = s_rint((data_ctr_num, data_ctr_denom),  effect=0.5,  seed=42)   # +0.5 clicks/user

print("=== ConstantEffectSizer ===")

delta = cont_eff.mean() - data_cont.mean()
print(f"continuous   before={data_cont.mean():.3f}   after={cont_eff.mean():.3f}  Δ={delta:+.3f}")

delta = prop_eff.mean() - data_prop.mean()
print(f"proportion   before={data_prop.mean():.4f}   after={prop_eff.mean():.4f}   Δ={delta:+.4f}")

before_avg = data_avg_num.sum() / data_avg_denom.sum()
after_avg  = rfl_eff[0].sum()   / rfl_eff[1].sum()
print(f"ratio float  before={before_avg:.2f}   after={after_avg:.2f}   Δ={after_avg - before_avg:+.2f}")

before_ctr = data_ctr_num.sum() / data_ctr_denom.sum()
after_ctr  = rint_eff[0].sum()  / rint_eff[1].sum()
print(f"ratio int    before={before_ctr:.5f}  after={after_ctr:.5f}  Δ={after_ctr - before_ctr:+.5f}")

=== ConstantEffectSizer ===
continuous   before=99.693   after=104.693  Δ=+5.000
proportion   before=0.1006   after=0.1206   Δ=+0.0200
ratio float  before=504.32   after=520.96   Δ=+16.64
ratio int    before=0.05065  after=0.10335  Δ=+0.05271


## `PercentEffectSizer` — relative (multiplicative) effect

`PercentEffectSizer` scales the metric by `(1 + effect)`. `effect=0.05` means
**+5% of the current value**.

| Metric type | What happens | Typical framing |
|---|---|---|
| Continuous iid | Every value `× (1 + E)` | "Revenue grows by **5%**" |
| Proportion iid | Rate shifts by `current_rate × E` (not by `E` itself) | "CTR grows by **10% of current CTR**" |
| Ratio float | Numerator `× (1 + E)`; denominator unchanged | "Average check grows by **5%**" |
| Ratio int / CTR | Numerator `× (1 + E)`, stochastically rounded; clipped at `denom` | "Clicks per user grow by **5%**" |
  
!!! warning "⚠️ Proportion pitfall."
    `PercentEffectSizer(effect=0.05)` on a 10% base rate produces:
    `0.10 + 0.10 × 0.05 = 0.105` — an absolute shift of **+0.5 pp**, not +5 pp.  
    If you mean *+5 percentage points*, use `ConstantEffectSizer(effect=0.05)`.
  
!!! tip "Usage"
    To use the effect sizer with given effect you should just pass `effect_sizer='percent'` and `effect=your_effect` to designer. Below we will just look at examples, reproducing of this code in simulations not needed.

In [3]:
s_cont_p = PercentEffectSizer(proportion=False, ratio=False,   seed=42)
s_prop_p = PercentEffectSizer(proportion=True,  ratio=False,   seed=42)
s_rfl_p  = PercentEffectSizer(proportion=False, ratio='float', seed=42)
s_rint_p = PercentEffectSizer(proportion=True,  ratio='int',   seed=42)

E = 0.05   # +5 %

cont_eff_p = s_cont_p(data_cont,                       effect=E, seed=42)
prop_eff_p = s_prop_p(data_prop,                       effect=E, seed=42)
rfl_eff_p  = s_rfl_p( (data_avg_num, data_avg_denom),  effect=E, seed=42)
rint_eff_p = s_rint_p((data_ctr_num, data_ctr_denom),  effect=E, seed=42)

print("=== PercentEffectSizer, effect=0.05 (+5 %) ===")

delta = cont_eff_p.mean() - data_cont.mean()
rel   = cont_eff_p.mean() / data_cont.mean() - 1
print(f"continuous   before={data_cont.mean():.3f}  after={cont_eff_p.mean():.3f}  Δ={delta:+.3f}  ({rel:+.2%})")

delta = prop_eff_p.mean() - data_prop.mean()
expected_delta = data_prop.mean() * E
print(f"proportion   before={data_prop.mean():.5f}  after={prop_eff_p.mean():.5f}  "
      f"Δ={delta:+.5f}  (expected ≈ {expected_delta:+.5f}  ← NOT +5 pp)")

before_avg = data_avg_num.sum() / data_avg_denom.sum()
after_avg  = rfl_eff_p[0].sum() / rfl_eff_p[1].sum()
print(f"ratio float  before={before_avg:.2f}  after={after_avg:.2f}  relative Δ={after_avg/before_avg - 1:+.2%}")

before_ctr = data_ctr_num.sum() / data_ctr_denom.sum()
after_ctr  = rint_eff_p[0].sum()/ rint_eff_p[1].sum()
print(f"ratio int    before={before_ctr:.5f}  after={after_ctr:.5f}  relative Δ={after_ctr/before_ctr - 1:+.2%}")


=== PercentEffectSizer, effect=0.05 (+5 %) ===
continuous   before=99.693  after=104.677  Δ=+4.985  (+5.00%)
proportion   before=0.10060  after=0.10560  Δ=+0.00500  (expected ≈ +0.00503  ← NOT +5 pp)
ratio float  before=504.32  after=529.54  relative Δ=+5.00%
ratio int    before=0.05065  after=0.05306  relative Δ=+4.77%


## Choosing the right EffectSizer

| If you say… | Use | `effect=` value |
|---|---|---|
| "Revenue grows by **$5 per user**" | `ConstantEffectSizer` | `5.0` |
| "Conversion grows by **+2 percentage points**" | `ConstantEffectSizer` | `0.02` |
| "Revenue grows by **5%**" | `PercentEffectSizer` | `0.05` |
| "CTR grows by **10% of current CTR**" (5% → 5.5%) | `PercentEffectSizer` | `0.10` |
| "Heavy users respond more than light users" | `FunctionEffectSizer` | heterogeneous |
| "You have holdout / model predictions" | `CustomEffectSizer` | custom |

### Quick decision rule

1. **Absolute shift** (same unit as the metric) → `ConstantEffectSizer`
2. **Relative lift** (percentage of current value) → `PercentEffectSizer`
3. **Heterogeneous / non-linear lift** (depends on the user) → `FunctionEffectSizer`
4. **Pre-computed uplift from a model or holdout** → `CustomEffectSizer`

### Worked example — CTR

Suppose baseline CTR is **5%** and you expect the feature to lift it by **10% relatively**
(i.e. 5% → 5.5%, a gain of +0.5 percentage points).

- Wrong choice: `ConstantEffectSizer(effect=0.10)` — would add 10 percentage points → 15% CTR.
- Correct choice: `PercentEffectSizer(effect=0.10)` — multiplies the numerator by 1.10,
  which shifts the rate by `rate × 0.10 = 0.005` (+0.5 pp). ✓

### Practical guideline

!!! tip "Advice"
    When in doubt, frame your effect hypothesis as _"the metric value for an average treated  
    user changes by X"_. If X is in the same units as the metric → **Constant**. If X is  
    expressed as a fraction of the current value → **Percent**.  
  

## `FunctionEffectSizer` — Heterogeneous effects

Use `FunctionEffectSizer` when the treatment lift **depends on each user's own baseline value** — e.g. heavy users respond differently from light users, or you want to cap the effect for outliers.

### Callable signature

```python
    def my_effect(
        test,             # test-group values (read-only numpy view)
        control,          # control-group values (read-only numpy view)
        test_control,     # full combined array (read-only numpy view)
        test_idx,         # integer indices of test rows in test_control
        control_idx,      # integer indices of control rows in test_control
        test_control_idx, # indices of the current simulation draw in raw_data
        seed,             # current simulation seed (int)
    ) -> new_test         # numpy array with the modified test values
```
  
!!! info "Applying method"
    You should define function with arguments above that returns test group target with effect.  
    For iid metrik it is array of shape test with test data + effect.  
    **Key feature:** function calls inside simulation. So exact simulation test and control would be given to the arguments.  
  
> **All input arrays are read-only.** You must return a *new* array; modifying `test` in place will raise an error.

### When to use

| Scenario | Example |
|---|---|
| Lift scales with usage intensity | Top-30% spenders get +10%, rest +3% |
| Capped absolute effect | Effect = min($5, 20% of value) |
| Segment-specific constants | Mobile users +2%, Desktop users +1% |
| Any non-linear transformation | Log-scale uplift |


In [6]:
import numpy as np
from mcab import DesignerIid, AaDataIid

# ── Heterogeneous lift: top-30% users get +10%, bottom-70% get +3% ──────────
def heterogeneous_lift(test, control, test_control,
                       test_idx, control_idx, test_control_idx, seed):
    new_test = test.copy()                   # always work on a copy
    threshold = np.percentile(test, 70)      # 70th percentile within test group
    heavy = test >= threshold                # top-30% mask
    new_test[heavy]  = test[heavy]  * 1.10  # +10% for heavy users
    new_test[~heavy] = test[~heavy] * 1.03  # +3%  for light users
    return new_test

# ── Wire into DesignerIid via effect_sizer='function' + func= ───────────────
designer = DesignerIid(
    target=data_cont,
    effect_sizer='function',
    func=heterogeneous_lift,
)

# ── Demonstrate the effect directly on a manual split ───────────────────────
rng = np.random.default_rng(42)
n = len(data_cont) // 2
idx = rng.choice(len(data_cont), size=n * 2, replace=False)
control_vals = data_cont[idx[:n]]
test_vals    = data_cont[idx[n:]]

# Apply the function as the Designer would at simulation time
lifted = heterogeneous_lift(
    test=test_vals,
    control=control_vals,
    test_control=data_cont[idx],
    test_idx=np.arange(n, 2 * n),
    control_idx=np.arange(n),
    test_control_idx=idx,
    seed=42,
)

threshold = np.percentile(test_vals, 70)
heavy_mask = test_vals >= threshold

print("=== FunctionEffectSizer — heterogeneous lift ===")
print(f"Control mean        : {control_vals.mean():.4f}")
print(f"Test (before lift)  : {test_vals.mean():.4f}")
print(f"Test (after lift)   : {lifted.mean():.4f}")
print(f"Overall lift        : {lifted.mean() / test_vals.mean() - 1:.2%}")
print()
print(f"Heavy users (top-30%)  after lift: {lifted[heavy_mask].mean():.4f}  (×1.10)")
print(f"Light users (bot-70%)  after lift: {lifted[~heavy_mask].mean():.4f}  (×1.03)")
print(f"Expected blended lift ≈ 0.30×10% + 0.70×3% = {0.30*0.10 + 0.70*0.03:.2%}")


=== FunctionEffectSizer — heterogeneous lift ===
Control mean        : 99.7181
Test (before lift)  : 99.6669
Test (after lift)   : 105.4831
Overall lift        : 5.84%

Heavy users (top-30%)  after lift: 148.0371  (×1.10)
Light users (bot-70%)  after lift: 87.2456  (×1.03)
Expected blended lift ≈ 0.30×10% + 0.70×3% = 5.10%


## `CustomEffectSizer` — pre-computed data with effect

Use this sizer when you have some intuition what target value should be reached for each observation after treatment.   
Precompute this value for each observation in dataset and give it as `custom_data` argument. 
If unit in simulation will be included in test group – your custom value will given, raw data otherwize.
  
- `custom_data` — the same observations with the effect already applied
  (e.g. from a holdout experiment, a causal uplift model, or a manual
  domain-expert transformation)
  
!!! info "Difference from 'function' sizer"
    Function sizer allows you compute any data during the simulation.  
    CustomEffect sizer uses **same** precomputed data in **each simulation**.  
  
### Constraints enforced at construction

| Condition | Checked |
|---|---|
| `raw_data.shape == custom_data.shape` | ✅ |
| `proportion=True, ratio=False` → `custom_data` contains only 0 and 1 | ✅ |
| `proportion=True, ratio=True` → `num ≤ denom` in both arrays | ✅ |


In [7]:
# ── CustomEffectSizer example ─────────────────────────────────────────────────
# Simulate a causal model output: per-user log-normal treatment multiplier
raw = RandomData(seed=42).normal_data(size=10_000, mean=100, std=30)

rng_model = np.random.default_rng(7)
multiplier = rng_model.lognormal(mean=np.log(1.05), sigma=0.05, size=len(raw))
custom = raw * multiplier          # the "model predicted" treated values

sizer = CustomEffectSizer(
    proportion=False,
    ratio=False,
    seed=42,
    custom_data=custom,
    raw_data=raw,
)

# Demonstrate: select first 500 users as a mock test group
idx = np.arange(500)
no_eff   = sizer(idx, effect=False)   # returns raw[idx]
with_eff = sizer(idx, effect=True)    # returns custom[idx]

print("CustomEffectSizer — per-user model predictions")
print(f"  without effect : mean={no_eff.mean():.3f}")
print(f"  with effect    : mean={with_eff.mean():.3f}")
print(f"  relative lift  : {(with_eff.mean() / no_eff.mean() - 1)*100:+.2f}%")

# ── Wire into a Designer (power methods are covered in section 6) ─────────────
target_custom = AaDataIid(raw)

designer_custom = DesignerIid(
    target_custom,
    effect_sizer='custom',
    custom_data=custom,
    seed=42,
)

print(f"\nDesignerIid with CustomEffectSizer: {type(designer_custom.add_effect).__name__}")


CustomEffectSizer — per-user model predictions
  without effect : mean=99.606
  with effect    : mean=103.985
  relative lift  : +4.40%

DesignerIid with CustomEffectSizer: CustomEffectSizer


!!! warning "Data Type"
    In both of `FunctionEffectSizer` and `CustomEffectSizer` you should return not effect but data + effect. Returning just effect would be an error because sizers should return sized test group metrik, not effect without baseline.

## Summary

### EffectSizer cheatsheet

| | Continuous iid | Proportion iid | Ratio float | Ratio int / CTR |
|---|---|---|---|---|
| **`ConstantEffectSizer(E)`** | each value `+ E` | rate `+ E` (bits flipped to match) | numerator `+ E` | numerator `+ E`, stochastic-rounded, clipped at denom |
| **`PercentEffectSizer(E)`** | each value `× (1+E)` | rate `+ rate × E` (bits flipped) | numerator `× (1+E)` | numerator `× (1+E)`, stochastic-rounded, clipped |
| **`FunctionEffectSizer`** | arbitrary callable | arbitrary callable | arbitrary callable | arbitrary callable |
| **`CustomEffectSizer`** | pre-computed rows | pre-computed rows | pre-computed rows | pre-computed rows |

> **Stochastic rounding** (Ratio int): because numerators are integers, fractional increments
> are applied probabilistically — e.g. `+0.05` events means each row gets `+1` with
> probability 0.05, `+0.0` otherwise. This preserves the expected effect without
> systematically biasing integer counts.

> **Proportion clip** (Ratio with `proportion=True`): after adding the effect the numerator
> is clipped at the denominator so that the implied rate never exceeds 1.

### Key takeaways

- **`ConstantEffectSizer`** is the right tool when your MDE is expressed in absolute units.
- **`PercentEffectSizer`** is the right tool when your MDE is a relative lift — but remember
  that for *proportion* metrics the shift in rate is `rate × E`, not `E` itself.
- **`FunctionEffectSizer`** unlocks heterogeneous treatment effects without leaving the
  simulation framework.
- **`CustomEffectSizer`** lets you replay real holdout uplifts or model predictions inside
  a Monte Carlo power study.
- All sizers are passed via the `effect_sizer=` parameter of any `Designer*` class and
  are fully composable with seeds, sample sizes, and ratio metrics covered in other sections.
